# Leveaniemi Process-Water Forecasting

This notebook is structured as two separate thesis methods.

**Method 1, deterministic consultant formula, comes first and can be run on its own.** It reconstructs the consultant's Excel mass-balance recurrence, runs LK/GK/GL scenario forecasts, and validates the 2020-2025 LK period against the observed data in `parameters_used.xlsx`.

**Method 2, machine-learning surrogate, is optional.** Run it only if you also want a Random Forest model that learns the consultant's recurrence and produces a second comparison. You can stop after Method 1 and still have a complete consultant-approach result.

**Important context:** the GK/GL values inside the consultant workbook are not treated as measured current/future truth. They are consultant-derived scenario/proxy values. LK, Leveaniemi-Kiruna, was not directly considered by the consultant, so this notebook infers an LK proxy from the available Leveaniemi and Kiruna leaching-rate structure. Future forecasts should use user-supplied new ore production/leaching inputs when available, or otherwise be reported as sensitivity analysis.

The workbook column `J` is treated as an inflow/pit-pump concentration. The recurrence output used for reconstruction is the consultant's GM result column `AI`, while `AF` is retained as the storage/state concentration that is fed forward. If your thesis definition of the target should instead be a different column, change `TARGET_COLUMN` in the configuration cell.


## Dependencies

For Method 1 only, install the core workbook and plotting packages if the imports below fail:

```python
%pip install pandas numpy openpyxl matplotlib xlsxwriter
```

For optional Method 2, install scikit-learn before running the ML section:

```python
%pip install scikit-learn
```


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import math
import warnings

REQUIRED_MODULES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}
missing = [package for module, package in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing required packages: " + ", ".join(missing) +
        ". Install them with `%pip install pandas numpy openpyxl matplotlib xlsxwriter`."
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")


def _mean_absolute_error(y_true, y_pred) -> float:
    values = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if values.empty:
        return np.nan
    return float((values["y_true"] - values["y_pred"]).abs().mean())


def _r2_score(y_true, y_pred) -> float:
    values = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if len(values) < 2:
        return np.nan
    residual_sum = float(((values["y_true"] - values["y_pred"]) ** 2).sum())
    total_sum = float(((values["y_true"] - values["y_true"].mean()) ** 2).sum())
    if np.isclose(total_sum, 0.0):
        return np.nan
    return 1.0 - residual_sum / total_sum


In [ ]:
# -----------------------------
# Project configuration
# -----------------------------
WORKBOOK_NAME = "Leveaniemi_data.xlsx"

def find_workbook(filename: str = WORKBOOK_NAME) -> Path:
    candidates = [Path(filename), Path.cwd() / filename]
    candidates.extend(parent / filename for parent in Path.cwd().resolve().parents)
    candidates.extend([
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        Path("/home/ojaami/dev/water-quality-prediction") / filename,
    ])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    try:
        from google.colab import files
    except Exception:
        files = None

    if files is not None:
        print(f"{filename} was not found in the Colab runtime. Upload the Excel file now.")
        uploaded = files.upload()
        exact_path = Path.cwd() / filename
        if exact_path.exists():
            return exact_path.resolve()
        excel_uploads = [Path.cwd() / name for name in uploaded if name.lower().endswith((".xlsx", ".xlsm", ".xls"))]
        if excel_uploads:
            print(f"Using uploaded workbook: {excel_uploads[0].name}")
            return excel_uploads[0].resolve()

    searched = "\n".join(str(candidate) for candidate in unique_candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Current notebook working directory is {Path.cwd()}. "
        f"If you are using Colab, upload {filename} to /content or mount Drive. Searched:\n{searched}"
    )

WORKBOOK_PATH = find_workbook()
print(f"Using workbook: {WORKBOOK_PATH}")
PROCESS_WATER_SHEET = "Process water"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "gm_output_conc"       # consultant's GM recurrence result in column AI
STATE_COLUMN = "prev_storage_conc"     # concentration fed into the next recurrence step
TRAIN_END_YEAR = 2025.5
ACTUAL_DATA_START_YEAR = 2020.0       # monitoring data before 2020 is not treated as actual observations
ACTUAL_DATA_END_YEAR = 2025.0
FORECAST_START_YEAR = 2026.0
FORECAST_END_YEAR = 2030.0
FORECAST_YEARS = np.round(np.arange(FORECAST_START_YEAR, FORECAST_END_YEAR + 0.01, 0.5), 1)

RANDOM_STATE = 42
N_MONTE_CARLO = 500        # thesis requirement: >= 200
RF_N_ESTIMATORS = 100      # 300 is usually enough for this small workbook; increase to 800 for a slower final sensitivity check.
LEACH_PERTURBATION = 0.15  # +/-15% uniform perturbation on leaching load/rate proxy

# Optional concentration limits. Fill these if the permit/project thresholds are known.
# The units must match PARAM_BLOCKS below.
ELEMENT_LIMITS = {
    "Cu": None,
    "NH4": None,
    "Cl": None,
    "Ni": None,
    "Zn": None,
    "Co": None,
    "Mo": None,
    "SO4": None,
    "Ca": None,
    "NO3": None,
    "PO4P": None,
    "As": None,
    "Cr": None,
}

# IMPORTANT: GK/GL columns in the workbook are consultant scenario/proxy values,
# not measured current/future ore-plan truth. LK is not in the workbook directly;
# it is inferred from the available Leveaniemi and Kiruna rate structure.
# Current operations can use time-varying LK ratios, such as 50/50 or 40/60.
# These fallback values are useful for sensitivity only.
# If real 2026-2030 ore inputs are available, replace None with a DataFrame/list of
# rows containing: parameter, year, production_mton, process_leach.
# Optional columns: ore_frac_gm, ore_frac_gk, ore_frac_gl, ore_frac_lk,
# lk_leveaniemi_frac, lk_kiruna_frac.
NEW_ORE_INPUTS = None

# Optional proxy schedule for varying ore mixes over time. Fill this when the mine
# uses different LK ratios in different years or seasons. Example:
# TIME_VARYING_ORE_MIX = pd.DataFrame([
#     {"year": 2026.0, "lk": 1.0, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
#     {"year": 2026.5, "lk": 1.0, "lk_leveaniemi": 0.40, "lk_kiruna": 0.60},
# ])
TIME_VARYING_ORE_MIX = None

# Fallback sensitivity scenarios when NEW_ORE_INPUTS is None.
# Fractions are normalized by row, so values only need to be proportional.
ORE_COMPONENTS = ["gm", "gk", "gl", "lk"]
LK_INTERNAL_COMPONENTS = ["lk_leveaniemi", "lk_kiruna"]
LK_DEFAULT_LEVEANIEMI_FRACTION = 0.50
LK_DEFAULT_KIRUNA_FRACTION = 0.50
ORE_MIX_SCENARIOS = {
    "GK100": {"gm": 0.0, "gk": 1.0, "gl": 0.0, "lk": 0.0},
    "GL100": {"gm": 0.0, "gk": 0.0, "gl": 1.0, "lk": 0.0},
    "LK50_50": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
    "LK40_60": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.40, "lk_kiruna": 0.60},
    "LK60_40": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.60, "lk_kiruna": 0.40},
    "GK60_GL40": {"gm": 0.0, "gk": 0.60, "gl": 0.40, "lk": 0.0},
    "GK50_LK50": {"gm": 0.0, "gk": 0.50, "gl": 0.0, "lk": 0.50, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
    "GL50_LK50": {"gm": 0.0, "gk": 0.0, "gl": 0.50, "lk": 0.50, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
}
SELECTED_ORE_MIX_SCENARIO = TIME_VARYING_ORE_MIX if TIME_VARYING_ORE_MIX is not None else "LK50_50"
SELECTED_ORE_MIX_LABEL = "TIME_VARYING_ORE_MIX" if TIME_VARYING_ORE_MIX is not None else SELECTED_ORE_MIX_SCENARIO
FORECAST_INPUT_MODE = (
    "user_supplied_new_ore_inputs" if NEW_ORE_INPUTS is not None
    else "time_varying_proxy_ore_mix" if TIME_VARYING_ORE_MIX is not None
    else "consultant_proxy_sensitivity"
)

PARAM_BLOCKS = {
    # Original thesis focus parameters.
    "Cu":   {"unit": "ug/l", "data_start_excel_row": 43,  "max_rows": 35, "observed_column": "Cu"},
    "NH4":  {"unit": "mg/l", "data_start_excel_row": 82,  "max_rows": 35, "observed_column": "NH4"},
    "Cl":   {"unit": "mg/l", "data_start_excel_row": 122, "max_rows": 35, "observed_column": "Cl"},
    "Ni":   {"unit": "ug/l", "data_start_excel_row": 161, "max_rows": 35, "observed_column": "Ni"},

    # Additional consultant workbook blocks with final recurrence outputs in column AI.
    "Zn":   {"unit": "ug/l", "data_start_excel_row": 200, "max_rows": 35, "observed_column": "Zn"},
    "Co":   {"unit": "ug/l", "data_start_excel_row": 240, "max_rows": 35, "observed_column": "Co"},
    "Mo":   {"unit": "ug/l", "data_start_excel_row": 279, "max_rows": 35, "observed_column": "Mo"},
    "SO4":  {"unit": "mg/l", "data_start_excel_row": 321, "max_rows": 35, "observed_column": "SO4"},
    "Ca":   {"unit": "mg/l", "data_start_excel_row": 361, "max_rows": 35, "observed_column": "Ca"},
    "NO3":  {"unit": "mg/l", "data_start_excel_row": 397, "max_rows": 35, "observed_column": "NO3"},
    "PO4P": {"unit": "mg/l", "data_start_excel_row": 433, "max_rows": 35, "observed_column": "PO4P"},
    "As":   {"unit": "ug/l", "data_start_excel_row": 507, "max_rows": 35, "observed_column": "As"},
    "Cr":   {"unit": "ug/l", "data_start_excel_row": 544, "max_rows": 35, "observed_column": "Cr"},
}

# Fluoride/F exists as an input-style block in the workbook, but the Process water
# sheet does not contain the same final AI recurrence output for it, so it is not
# included in the modelled parameter list.
PLOT_COLUMNS = 3


def make_parameter_axes(n_parameters: int, panel_width: float = 5.6, panel_height: float = 3.4, sharex: bool = True):
    ncols = min(PLOT_COLUMNS, max(1, n_parameters))
    nrows = math.ceil(n_parameters / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_width * ncols, panel_height * nrows), sharex=sharex)
    axes_all = np.atleast_1d(axes).ravel()
    for ax in axes_all[n_parameters:]:
        ax.set_visible(False)
    return fig, axes_all[:n_parameters]


## Workbook Extraction

The extraction uses explicit zero-indexed Excel columns so the logic remains visible. The column names below mirror the consultant's sheet: production/leaching alternatives, pit-pump input, water-balance flows, storage volume, storage state, and GM recurrence result.

In [ ]:
COLUMN_MAP = {
    "year": 0,
    "tailings_load": 1,
    "roll_leach_rate": 2,
    "production_mton_gm": 3,
    "process_leach_gm": 4,
    "production_mton_gk": 5,
    "process_leach_gk": 6,
    "process_leach_gl": 7,
    "pit_pump_volume": 8,
    "pit_pump_conc": 9,
    "ditch_sw_flow": 10,
    "ditch_sw_conc": 11,
    "ditch_se_flow": 12,
    "ditch_se_conc": 13,
    "gruvberget_flow": 14,
    "gruvberget_conc": 15,
    "surface_water_flow": 16,
    "surface_water_conc": 17,
    "process_loss_flow": 18,
    "process_loss_conc": 19,
    "dams_from_process_flow": 20,
    "dams_from_process_conc": 21,
    "p2_process_flow": 22,
    "p2_process_conc": 23,
    "tailings_storage_flow": 24,
    "tailings_storage_conc": 25,
    "discharge_flow": 26,
    "discharge_conc": 27,
    "leakage_flow": 28,
    "leakage_conc": 29,
    "storage_volume": 30,
    "storage_conc_state_col_af": 31,
    "losses": 32,
    "gains": 33,
    "gm_output_conc": 34,
}

GAIN_FLOW_COLUMNS = [
    "pit_pump_volume",
    "ditch_sw_flow",
    "ditch_se_flow",
    "gruvberget_flow",
    "surface_water_flow",
]
LOSS_FLOW_COLUMNS = [
    "process_loss_flow",
    "tailings_storage_flow",
    "discharge_flow",
    "leakage_flow",
]
LOAD_PAIRS = [
    ("pit_pump_volume", "pit_pump_conc"),
    ("ditch_sw_flow", "ditch_sw_conc"),
    ("ditch_se_flow", "ditch_se_conc"),
    ("gruvberget_flow", "gruvberget_conc"),
    ("surface_water_flow", "surface_water_conc"),
    ("process_loss_flow", "process_loss_conc"),
    ("tailings_storage_flow", "tailings_storage_conc"),
    ("discharge_flow", "discharge_conc"),
    ("leakage_flow", "leakage_conc"),
]

def _numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _clean_proxy_rate(series: pd.Series) -> pd.Series:
    cleaned = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    median = cleaned.median(skipna=True)
    fill_value = median if np.isfinite(median) else 0.0
    return cleaned.fillna(fill_value)


def add_lk_proxy_inputs(extracted: pd.DataFrame) -> pd.DataFrame:
    """Infer an LK, Leveaniemi-Kiruna, process-leaching proxy from workbook rates.

    The consultant workbook has GK and GL proxy columns, but no direct LK column.
    This derives Leveaniemi and Kiruna rates from those formulas. The actual
    LK ratio can then be set as 50/50, 40/60, or any time-varying schedule.
    Replace this with NEW_ORE_INPUTS if measured LK data exists.
    """
    out = extracted.copy()
    gm_rate = out["roll_leach_rate"]
    prod_gm = out["production_mton_gm"].replace(0, np.nan)
    prod_gk = out["production_mton_gk"].replace(0, np.nan)

    kiruna_rate = (out["process_leach_gk"] - gm_rate * 0.4 * out["production_mton_gm"]) / (0.6 * prod_gk)
    leveaniemi_rate = ((out["process_leach_gl"] / prod_gm) - gm_rate * 0.4) / 0.6

    out["kiruna_leach_rate_proxy"] = _clean_proxy_rate(kiruna_rate)
    out["leveaniemi_leach_rate_proxy"] = _clean_proxy_rate(leveaniemi_rate)
    out["production_mton_lk"] = out["production_mton_gm"]
    out["lk_leveaniemi_frac"] = LK_DEFAULT_LEVEANIEMI_FRACTION
    out["lk_kiruna_frac"] = LK_DEFAULT_KIRUNA_FRACTION
    out["process_leach_lk"] = (
        out["lk_leveaniemi_frac"] * out["leveaniemi_leach_rate_proxy"] +
        out["lk_kiruna_frac"] * out["kiruna_leach_rate_proxy"]
    ) * out["production_mton_lk"]
    return out


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create compact mass-balance features that help the RF learn the recurrence."""
    out = df.copy()

    out[GAIN_FLOW_COLUMNS + LOSS_FLOW_COLUMNS] = out[GAIN_FLOW_COLUMNS + LOSS_FLOW_COLUMNS].fillna(0.0)
    out["gross_gain_flow"] = out[GAIN_FLOW_COLUMNS].sum(axis=1)
    out["gross_loss_flow"] = out[LOSS_FLOW_COLUMNS].sum(axis=1)
    out["net_flow"] = out["gross_gain_flow"] - out["gross_loss_flow"]

    load_terms = []
    for flow_col, conc_col in LOAD_PAIRS:
        term_name = f"{flow_col}_load"
        out[term_name] = out[flow_col].fillna(0.0) * out[conc_col].fillna(0.0)
        load_terms.append(term_name)
    out["known_water_load_proxy"] = out[load_terms].sum(axis=1)

    out["production_mton"] = out["production_mton"].replace(0, np.nan)
    out["storage_volume"] = out["storage_volume"].replace(0, np.nan)
    out["leach_per_mton"] = out["process_leach"] / out["production_mton"]
    out["leach_per_storage_volume"] = out["process_leach"] / out["storage_volume"]
    out["storage_mass_proxy"] = out["storage_volume"] * out[STATE_COLUMN]
    out["pump_conc_x_volume"] = out["pit_pump_volume"].fillna(0.0) * out["pit_pump_conc"].fillna(0.0)
    return out


def extract_parameter_block(process_water: pd.DataFrame, parameter: str, spec: dict) -> pd.DataFrame:
    start = spec["data_start_excel_row"] - 1
    raw = process_water.iloc[start:start + spec["max_rows"], :].copy()

    extracted = pd.DataFrame({name: _numeric(raw.iloc[:, idx]) for name, idx in COLUMN_MAP.items()})
    extracted["source_excel_row"] = raw.index + 1
    extracted = extracted[extracted["year"].between(2014.0, 2030.5, inclusive="both")].copy()
    extracted = extracted.sort_values("year").reset_index(drop=True)

    extracted["parameter"] = parameter
    extracted["unit"] = spec["unit"]
    extracted["calendar_year"] = np.floor(extracted["year"]).astype(int)
    extracted["half_year"] = np.isclose(extracted["year"] % 1, 0.5).astype(int)
    extracted["year_index"] = extracted["year"] - extracted["year"].min()
    extracted["tailings_load"] = extracted["tailings_load"].fillna(0.0)

    # AI is the consultant's GM result. If the terminal row lacks AI, keep AF as a fallback only.
    extracted["gm_output_conc"] = extracted["gm_output_conc"].fillna(extracted["storage_conc_state_col_af"])

    # The recurrence state for row t is the previous modelled output. Seed the first row from AF.
    extracted[STATE_COLUMN] = extracted["gm_output_conc"].shift(1)
    if not extracted.empty:
        extracted.loc[0, STATE_COLUMN] = extracted.loc[0, "storage_conc_state_col_af"]
    extracted["is_initial_row"] = False
    if not extracted.empty:
        extracted.loc[0, "is_initial_row"] = True

    # Historical training uses the original GM-heavy assumption.
    extracted["ore_frac_gm"] = 1.0
    extracted["ore_frac_gk"] = 0.0
    extracted["ore_frac_gl"] = 0.0
    extracted["ore_frac_lk"] = 0.0
    extracted = add_lk_proxy_inputs(extracted)
    extracted["production_mton"] = extracted["production_mton_gm"]
    extracted["process_leach"] = extracted["process_leach_gm"]

    return add_derived_features(extracted)


process_water = pd.read_excel(WORKBOOK_PATH, sheet_name=PROCESS_WATER_SHEET, header=None, engine="openpyxl")
blocks = [extract_parameter_block(process_water, parameter, spec) for parameter, spec in PARAM_BLOCKS.items()]
model_data = pd.concat(blocks, ignore_index=True)

summary = model_data.groupby("parameter").agg(
    rows=("year", "count"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    first_output=(TARGET_COLUMN, "first"),
    last_output=(TARGET_COLUMN, "last"),
)
display(summary)
display(model_data.loc[:, ["parameter", "year", "half_year", "production_mton_gm", "process_leach_gm", "process_leach_gk", "process_leach_gl", "process_leach_lk", STATE_COLUMN, TARGET_COLUMN]].head(12))

## Ore-Input Substitution

The model learns from the consultant's calculated recurrence rows. For forecasting, use `NEW_ORE_INPUTS` when real new ore production/leaching assumptions are available. If `NEW_ORE_INPUTS = None`, the notebook uses workbook GK/GL proxy columns and an inferred LK proxy as clearly labelled sensitivity scenarios, not as observed truth.

In [ ]:
def normalize_mix(mix: dict) -> dict:
    values = {component: float(mix.get(component, 0.0)) for component in ORE_COMPONENTS}
    total = sum(values.values())
    if total <= 0:
        raise ValueError("Ore mix fractions must sum to a positive value.")
    return {component: value / total for component, value in values.items()}


def normalize_lk_internal_mix(mix: dict) -> dict:
    values = {
        "lk_leveaniemi": float(mix.get("lk_leveaniemi", LK_DEFAULT_LEVEANIEMI_FRACTION)),
        "lk_kiruna": float(mix.get("lk_kiruna", LK_DEFAULT_KIRUNA_FRACTION)),
    }
    total = sum(values.values())
    if total <= 0:
        return {"lk_leveaniemi": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna": LK_DEFAULT_KIRUNA_FRACTION}
    return {component: value / total for component, value in values.items()}


def scenario_to_table(scenario: str | dict | pd.DataFrame, years=FORECAST_YEARS) -> pd.DataFrame:
    if isinstance(scenario, str):
        if scenario not in ORE_MIX_SCENARIOS:
            raise KeyError(f"Unknown ore-mix scenario: {scenario}")
        scenario_def = ORE_MIX_SCENARIOS[scenario]
        mix = normalize_mix(scenario_def)
        lk_mix = normalize_lk_internal_mix(scenario_def)
        return pd.DataFrame({"year": years, **mix, **lk_mix})

    if isinstance(scenario, dict):
        mix = normalize_mix(scenario)
        lk_mix = normalize_lk_internal_mix(scenario)
        return pd.DataFrame({"year": years, **mix, **lk_mix})

    table = scenario.copy()
    if "year" not in table.columns:
        raise ValueError("Scenario table is missing required column: year")
    for component in ORE_COMPONENTS:
        if component not in table.columns:
            table[component] = 0.0
    for component in LK_INTERNAL_COMPONENTS:
        if component not in table.columns:
            table[component] = LK_DEFAULT_LEVEANIEMI_FRACTION if component == "lk_leveaniemi" else LK_DEFAULT_KIRUNA_FRACTION
    fractions = table[ORE_COMPONENTS].astype(float)
    totals = fractions.sum(axis=1).replace(0, np.nan)
    table[ORE_COMPONENTS] = fractions.div(totals, axis=0)
    lk_fractions = table[LK_INTERNAL_COMPONENTS].astype(float)
    lk_totals = lk_fractions.sum(axis=1).replace(0, np.nan)
    table[LK_INTERNAL_COMPONENTS] = lk_fractions.div(lk_totals, axis=0).fillna({"lk_leveaniemi": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna": LK_DEFAULT_KIRUNA_FRACTION})
    return table[["year", *ORE_COMPONENTS, *LK_INTERNAL_COMPONENTS]]


def apply_new_ore_inputs(df: pd.DataFrame, ore_inputs) -> pd.DataFrame | None:
    """Apply measured/user-supplied forecast inputs instead of consultant proxy columns."""
    if ore_inputs is None:
        return None

    table = pd.DataFrame(ore_inputs).copy()
    required = {"parameter", "year", "production_mton", "process_leach"}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"NEW_ORE_INPUTS is missing required columns: {sorted(missing)}")

    ore_fraction_columns = [f"ore_frac_{component}" for component in ORE_COMPONENTS]
    lk_internal_columns = [f"{component}_frac" for component in LK_INTERNAL_COMPONENTS]
    for col in ore_fraction_columns:
        if col not in table.columns:
            table[col] = np.nan
    for col in lk_internal_columns:
        if col not in table.columns:
            table[col] = np.nan

    table = table[["parameter", "year", "production_mton", "process_leach", *ore_fraction_columns, *lk_internal_columns]].copy()
    table["year"] = pd.to_numeric(table["year"], errors="coerce")
    for col in ["production_mton", "process_leach", *ore_fraction_columns, *lk_internal_columns]:
        table[col] = pd.to_numeric(table[col], errors="coerce")
    table = table.rename(columns={"production_mton": "production_mton_new", "process_leach": "process_leach_new", **{col: f"{col}_new" for col in [*ore_fraction_columns, *lk_internal_columns]}})

    out = df.drop(columns=[*ore_fraction_columns, *lk_internal_columns], errors="ignore").copy()
    out = out.merge(table, on=["parameter", "year"], how="left")
    missing_rows = out[out["process_leach_new"].isna() | out["production_mton_new"].isna()][["parameter", "year"]]
    if not missing_rows.empty:
        raise ValueError("NEW_ORE_INPUTS does not cover all forecast rows:\n" + missing_rows.to_string(index=False))
    out["production_mton"] = out["production_mton_new"]
    out["process_leach"] = out["process_leach_new"]
    for col in ore_fraction_columns:
        out[col] = out[f"{col}_new"].fillna(0.0)
    for col in lk_internal_columns:
        default = LK_DEFAULT_LEVEANIEMI_FRACTION if col == "lk_leveaniemi_frac" else LK_DEFAULT_KIRUNA_FRACTION
        out[col] = out[f"{col}_new"].fillna(default)
    out = out.drop(columns=["production_mton_new", "process_leach_new", *[f"{col}_new" for col in [*ore_fraction_columns, *lk_internal_columns]]])
    return add_derived_features(out)


def apply_ore_mix(df: pd.DataFrame, scenario: str | dict | pd.DataFrame, ore_inputs=NEW_ORE_INPUTS) -> pd.DataFrame:
    out = df.copy()
    user_input_df = apply_new_ore_inputs(out, ore_inputs)
    if user_input_df is not None:
        return user_input_df

    mix_table = scenario_to_table(scenario, years=out["year"].to_numpy())
    ore_fraction_columns = [f"ore_frac_{component}" for component in ORE_COMPONENTS]
    lk_internal_columns = [f"{component}_frac" for component in LK_INTERNAL_COMPONENTS]
    out = out.drop(columns=[*ore_fraction_columns, *lk_internal_columns], errors="ignore")
    rename_map = {component: f"ore_frac_{component}" for component in ORE_COMPONENTS}
    rename_map.update({component: f"{component}_frac" for component in LK_INTERNAL_COMPONENTS})
    out = out.merge(mix_table.rename(columns=rename_map), on="year", how="left")
    out[ore_fraction_columns] = out[ore_fraction_columns].fillna({"ore_frac_gm": 1.0, "ore_frac_gk": 0.0, "ore_frac_gl": 0.0, "ore_frac_lk": 0.0})
    out[lk_internal_columns] = out[lk_internal_columns].fillna({"lk_leveaniemi_frac": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna_frac": LK_DEFAULT_KIRUNA_FRACTION})
    lk_total = out["lk_leveaniemi_frac"] + out["lk_kiruna_frac"]
    out["lk_leveaniemi_frac"] = np.where(lk_total > 0, out["lk_leveaniemi_frac"] / lk_total, LK_DEFAULT_LEVEANIEMI_FRACTION)
    out["lk_kiruna_frac"] = np.where(lk_total > 0, out["lk_kiruna_frac"] / lk_total, LK_DEFAULT_KIRUNA_FRACTION)

    # Consultant-proxy fallback only: workbook GK/GL and inferred LK values are not observed new ore inputs.
    # GL uses the same production schedule column as the workbook's GL leaching-load formula.
    production_mton_gl_proxy = out["production_mton_gm"]
    out["process_leach_lk_dynamic"] = (
        out["lk_leveaniemi_frac"] * out["leveaniemi_leach_rate_proxy"] +
        out["lk_kiruna_frac"] * out["kiruna_leach_rate_proxy"]
    ) * out["production_mton_lk"]
    out["production_mton"] = (
        out["ore_frac_gm"] * out["production_mton_gm"] +
        out["ore_frac_gk"] * out["production_mton_gk"] +
        out["ore_frac_gl"] * production_mton_gl_proxy +
        out["ore_frac_lk"] * out["production_mton_lk"]
    )
    out["process_leach"] = (
        out["ore_frac_gm"] * out["process_leach_gm"] +
        out["ore_frac_gk"] * out["process_leach_gk"] +
        out["ore_frac_gl"] * out["process_leach_gl"] +
        out["ore_frac_lk"] * out["process_leach_lk_dynamic"]
    )
    return add_derived_features(out)


selected_mix_table = scenario_to_table(SELECTED_ORE_MIX_SCENARIO)
selected_mix_table["input_mode"] = FORECAST_INPUT_MODE
display(selected_mix_table.head())

## Deterministic Consultant Formula Method

This is the non-ML version of the project. It directly implements the consultant's mass-balance recurrence from the Excel formulas: contaminant load from ore and water sources is converted into an incoming concentration, then mixed with the previous stored concentration. This gives a complete deterministic baseline that can be presented separately from the Random Forest surrogate.

The first table checks whether the Python formula reproduces the consultant workbook's GM output column. Small errors mean the mathematical reconstruction is behaving like the Excel model. The later tables use the same formula for LK/GK/GL scenario forecasting.


In [ ]:
# -----------------------------
# Deterministic consultant formula implementation
# -----------------------------
CONSULTANT_FORMULA_METHOD = "Deterministic consultant formula"
CONSULTANT_FORMULA_ROW_SHIFTED_PARAMETERS = set(PARAM_BLOCKS) - {"NH4"}
CONSULTANT_LEACH_DIVISOR = {
    parameter: (1000.0 if PARAM_BLOCKS[parameter]["unit"] == "mg/l" else 1.0)
    for parameter in PARAM_BLOCKS
}
CONSULTANT_TAILINGS_DIVISOR = {
    "Cu": 1.0,
    "Cl": 1000.0,
    "Ni": 1.0,
    "Zn": 1.0,
    "SO4": 1000.0,
    "Ca": 1000.0,
    "NO3": 1000.0,
    "PO4P": 1000.0,
}
CONSULTANT_ROLL_RATE_LOAD_PARAMETERS = {"Co"}


def _finite_float(value, default: float = 0.0) -> float:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if np.isfinite(number) else default


def consultant_formula_ore_load(row: pd.Series, process_leach_col: str = "process_leach") -> float:
    """Parameter-specific ore/tailings load term from the consultant formula."""
    parameter = row["parameter"]
    process_leach = _finite_float(row.get(process_leach_col, row.get("process_leach")))
    tailings_load = _finite_float(row.get("tailings_load"))

    leach_divisor = CONSULTANT_LEACH_DIVISOR.get(parameter, 1.0)
    ore_load = process_leach / leach_divisor

    tailings_divisor = CONSULTANT_TAILINGS_DIVISOR.get(parameter)
    if tailings_divisor is not None:
        ore_load += tailings_load / tailings_divisor
    if parameter in CONSULTANT_ROLL_RATE_LOAD_PARAMETERS:
        ore_load += _finite_float(row.get("roll_leach_rate"))
    return ore_load


def consultant_formula_step(row: pd.Series, previous_conc: float, process_leach_col: str = "process_leach") -> float:
    """One consultant recurrence step for one parameter and one half-year row."""
    pit_flow = _finite_float(row.get("pit_pump_volume"))
    ditch_sw_flow = _finite_float(row.get("ditch_sw_flow"))
    ditch_se_flow = _finite_float(row.get("ditch_se_flow"))
    gruvberget_flow = _finite_float(row.get("gruvberget_flow"))
    surface_water_flow = _finite_float(row.get("surface_water_flow"))
    process_loss_flow = _finite_float(row.get("process_loss_flow"))
    storage_volume = _finite_float(row.get("storage_volume"))
    previous_conc = _finite_float(previous_conc)

    gains = pit_flow + ditch_sw_flow + ditch_se_flow + gruvberget_flow + surface_water_flow
    effective_volume = gains + pit_flow - process_loss_flow
    mixing_volume = gains + storage_volume

    if effective_volume <= 0 or mixing_volume <= 0:
        return np.nan

    total_load = (
        consultant_formula_ore_load(row, process_leach_col=process_leach_col)
        + pit_flow * _finite_float(row.get("pit_pump_conc"))
        + ditch_sw_flow * _finite_float(row.get("ditch_sw_conc"))
        + ditch_se_flow * _finite_float(row.get("ditch_se_conc"))
        + gruvberget_flow * _finite_float(row.get("gruvberget_conc"))
        - process_loss_flow * _finite_float(row.get("process_loss_conc"))
    )
    incoming_conc = total_load / effective_volume
    prediction = (incoming_conc * gains + storage_volume * previous_conc) / mixing_volume
    return float(prediction) if np.isfinite(prediction) else np.nan


def reproduce_consultant_workbook_parameter(parameter: str, data: pd.DataFrame) -> pd.DataFrame:
    """Reproduce the consultant's GM Excel output column for one parameter block.

    The reproduction check uses the workbook's own AF storage-state values.
    That makes the check match the Excel formulas exactly, including workbook-specific
    state references in the chloride block. Forecasts still use sequential prediction
    feedback, because future AF values do not exist.
    """
    param_df = data[data["parameter"] == parameter].sort_values("year").reset_index(drop=True)
    records = []

    if parameter in CONSULTANT_FORMULA_ROW_SHIFTED_PARAMETERS:
        initial_state = _finite_float(param_df.loc[0, "storage_conc_state_col_af"], _finite_float(param_df.loc[0, TARGET_COLUMN]))
        first_prediction = _finite_float(param_df.loc[0, TARGET_COLUMN], initial_state)
        records.append({
            "method": CONSULTANT_FORMULA_METHOD,
            "parameter": parameter,
            "unit": param_df.loc[0, "unit"],
            "year": float(param_df.loc[0, "year"]),
            "formula_input_year": np.nan,
            "workbook_target": first_prediction,
            "formula_prediction": first_prediction,
            "formula_error": 0.0,
            "formula_alignment": "initial seed row",
        })

        for input_idx in range(len(param_df) - 1):
            input_row = param_df.loc[input_idx]
            output_row = param_df.loc[input_idx + 1]
            workbook_state = _finite_float(input_row["storage_conc_state_col_af"], _finite_float(input_row[TARGET_COLUMN]))
            prediction = consultant_formula_step(input_row, workbook_state, process_leach_col="process_leach_gm")
            target = _finite_float(output_row[TARGET_COLUMN], np.nan)
            records.append({
                "method": CONSULTANT_FORMULA_METHOD,
                "parameter": parameter,
                "unit": output_row["unit"],
                "year": float(output_row["year"]),
                "formula_input_year": float(input_row["year"]),
                "workbook_target": target,
                "formula_prediction": prediction,
                "formula_error": prediction - target,
                "formula_alignment": "previous workbook row input using AF state",
            })
    else:
        for _, row in param_df.iterrows():
            workbook_state = _finite_float(row["storage_conc_state_col_af"], _finite_float(row[TARGET_COLUMN]))
            prediction = consultant_formula_step(row, workbook_state, process_leach_col="process_leach_gm")
            target = _finite_float(row[TARGET_COLUMN], np.nan)
            records.append({
                "method": CONSULTANT_FORMULA_METHOD,
                "parameter": parameter,
                "unit": row["unit"],
                "year": float(row["year"]),
                "formula_input_year": float(row["year"]),
                "workbook_target": target,
                "formula_prediction": prediction,
                "formula_error": prediction - target,
                "formula_alignment": "current workbook row input using AF state",
            })

    return pd.DataFrame(records)

def consultant_formula_reproduction_metrics_table(reproduction: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (parameter, unit), group in reproduction.groupby(["parameter", "unit"], dropna=False):
        comparable = group.dropna(subset=["workbook_target", "formula_prediction"]).copy()
        rows.append({
            "method": CONSULTANT_FORMULA_METHOD,
            "parameter": parameter,
            "unit": unit,
            "rows_checked": len(comparable),
            "mae_vs_workbook": _mean_absolute_error(comparable["workbook_target"], comparable["formula_prediction"]) if len(comparable) else np.nan,
            "max_abs_error_vs_workbook": comparable["formula_error"].abs().max() if len(comparable) else np.nan,
            "r2_vs_workbook": _r2_score(comparable["workbook_target"], comparable["formula_prediction"]) if len(comparable) > 1 else np.nan,
        })
    return pd.DataFrame(rows)


def consultant_formula_monte_carlo(
    parameter: str,
    data: pd.DataFrame,
    scenario: str | dict | pd.DataFrame = SELECTED_ORE_MIX_SCENARIO,
    ore_inputs=NEW_ORE_INPUTS,
    prediction_years=None,
    n_runs: int = N_MONTE_CARLO,
    leach_perturbation: float = LEACH_PERTURBATION,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Forecast with the deterministic consultant formula and sequential state feedback."""
    param_df = data[data["parameter"] == parameter].sort_values("year").copy()
    prediction_years = FORECAST_YEARS if prediction_years is None else np.round(np.asarray(prediction_years, dtype=float), 1)
    forecast_template = param_df[param_df["year"].isin(prediction_years)].copy()
    if forecast_template.empty:
        raise ValueError(f"No consultant-formula template rows found for {parameter} and years {prediction_years}.")
    forecast_template = apply_ore_mix(forecast_template, scenario, ore_inputs=ore_inputs).sort_values("year")

    start_year = float(np.min(prediction_years))
    historical_state = param_df[(param_df["year"] < start_year) & param_df[TARGET_COLUMN].notna()].sort_values("year")
    if historical_state.empty:
        raise ValueError(f"Cannot start consultant formula recurrence for {parameter}; no previous state exists before {start_year}.")
    start_state = float(historical_state[TARGET_COLUMN].dropna().iloc[-1])
    input_mode = (
        "user_supplied_new_ore_inputs" if ore_inputs is not None
        else "time_varying_proxy_ore_mix" if isinstance(scenario, pd.DataFrame)
        else "consultant_proxy_sensitivity"
    )

    rng = np.random.default_rng(random_state + sum(ord(ch) for ch in parameter) + 2400)
    records = []
    for run in range(n_runs):
        state = start_state
        for _, template_row in forecast_template.iterrows():
            row = template_row.copy()
            row["process_leach"] = row["process_leach"] * rng.uniform(1.0 - leach_perturbation, 1.0 + leach_perturbation)
            prediction = consultant_formula_step(row, state, process_leach_col="process_leach")
            state = prediction
            records.append({
                "run": run,
                "method": CONSULTANT_FORMULA_METHOD,
                "input_mode": input_mode,
                "parameter": parameter,
                "unit": row["unit"],
                "year": float(row["year"]),
                "calendar_year": int(row["calendar_year"]),
                "half_year": int(row["half_year"]),
                "ore_frac_gm": float(row["ore_frac_gm"]),
                "ore_frac_gk": float(row["ore_frac_gk"]),
                "ore_frac_gl": float(row["ore_frac_gl"]),
                "ore_frac_lk": float(row["ore_frac_lk"]),
                "lk_leveaniemi_frac": float(row.get("lk_leveaniemi_frac", LK_DEFAULT_LEVEANIEMI_FRACTION)),
                "lk_kiruna_frac": float(row.get("lk_kiruna_frac", LK_DEFAULT_KIRUNA_FRACTION)),
                "prediction": prediction,
            })
    return pd.DataFrame(records)


def summarize_consultant_formula_simulations(simulations: pd.DataFrame) -> pd.DataFrame:
    group_columns = [
        "method", "parameter", "unit", "input_mode", "year", "calendar_year", "half_year",
        "ore_frac_gm", "ore_frac_gk", "ore_frac_gl", "ore_frac_lk",
        "lk_leveaniemi_frac", "lk_kiruna_frac",
    ]
    return (
        simulations
        .groupby(group_columns, dropna=False)["prediction"]
        .quantile([0.10, 0.50, 0.90])
        .unstack()
        .rename(columns={0.10: "p10", 0.50: "p50", 0.90: "p90"})
        .reset_index()
        .sort_values(["parameter", "year"])
    )


def plot_consultant_formula_forecast_panels(data: pd.DataFrame, forecast: pd.DataFrame) -> Path:
    fig, axes = make_parameter_axes(len(PARAM_BLOCKS), panel_height=3.6)

    for ax, parameter in zip(axes, PARAM_BLOCKS):
        hist = data[(data["parameter"] == parameter) & data["year"].between(ACTUAL_DATA_START_YEAR, TRAIN_END_YEAR, inclusive="both")].sort_values("year")
        fc = forecast[forecast["parameter"] == parameter].sort_values("year")
        unit = PARAM_BLOCKS[parameter]["unit"]

        ax.plot(hist["year"], hist[TARGET_COLUMN], color="#374151", marker="o", linewidth=1.6, markersize=4, label="Consultant workbook model, not observed")
        ax.fill_between(fc["year"].to_numpy(), fc["p10"].to_numpy(), fc["p90"].to_numpy(), color="#f9a8d4", alpha=0.32, label="Formula P10-P90")
        ax.plot(fc["year"], fc["p50"], color="#be185d", marker="o", linewidth=2.0, markersize=4, label="Formula P50")
        limit = ELEMENT_LIMITS.get(parameter)
        if limit is not None and np.isfinite(float(limit)):
            ax.axhline(float(limit), color="#b91c1c", linestyle=":", linewidth=1.6, label="Limit")
        ax.axvline(TRAIN_END_YEAR, color="#9ca3af", linestyle="--", linewidth=1.2)
        ax.set_title(f"{parameter} ({unit})")
        ax.set_ylabel(f"Concentration ({unit})")
        ax.grid(True, alpha=0.25)

    for ax in axes:
        ax.set_xlabel("Year")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
    input_label = "user-supplied new ore inputs" if NEW_ORE_INPUTS is not None else f"proxy ore mix: {SELECTED_ORE_MIX_LABEL}"
    fig.suptitle(f"Deterministic Consultant Formula Forecast ({input_label}) with ±15% Leaching Uncertainty", y=0.98, fontsize=15)
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))

    out_path = OUTPUT_DIR / "leveaniemi_consultant_formula_forecast_bands.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    return out_path


consultant_formula_reproduction = pd.concat(
    [reproduce_consultant_workbook_parameter(parameter, model_data) for parameter in PARAM_BLOCKS],
    ignore_index=True,
)
consultant_formula_reproduction_metrics = consultant_formula_reproduction_metrics_table(consultant_formula_reproduction)

consultant_formula_forecast_simulations = pd.concat(
    [consultant_formula_monte_carlo(parameter, model_data) for parameter in PARAM_BLOCKS],
    ignore_index=True,
)
consultant_formula_forecast_summary = summarize_consultant_formula_simulations(consultant_formula_forecast_simulations)
consultant_formula_annual_forecast_summary = consultant_formula_forecast_summary[np.isclose(consultant_formula_forecast_summary["year"] % 1, 0.0)].copy()
consultant_formula_figure_path = plot_consultant_formula_forecast_panels(model_data, consultant_formula_forecast_summary)

display(consultant_formula_reproduction_metrics)
display(consultant_formula_annual_forecast_summary)


## 2020-2025 Actual Data and LK Schedule

This shared section loads `parameters_used.xlsx`. Only the 2020-2025 values from this workbook are treated as actual observed monitoring data. Pre-2020 consultant workbook rows are model/formula rows, not observations.

The LK ore-mix schedule is used first by Method 1 and later by optional Method 2 if you choose to run it.


In [ ]:
# -----------------------------
# 2020-2025 hindcast/back-test configuration
# -----------------------------
PARAMETERS_WORKBOOK_NAME = "parameters_used.xlsx"
PARAMETERS_WORKBOOK_PATH = find_workbook(PARAMETERS_WORKBOOK_NAME)
print(f"Using parameter/validation workbook: {PARAMETERS_WORKBOOK_PATH}")

HINDCAST_START_YEAR = ACTUAL_DATA_START_YEAR
HINDCAST_END_YEAR = ACTUAL_DATA_END_YEAR
HINDCAST_MONTE_CARLO_RUNS = 300


def load_observed_decimal_date(path: Path) -> pd.DataFrame:
    """Load observed seasonal concentrations for all modelled parameters that appear in DECIMAL DATE."""
    observed = pd.read_excel(path, sheet_name="DECIMAL DATE", engine="openpyxl")
    observed = observed.rename(columns={
        "Period": "year",
        "NH4-N": "NH4",
        "NO3-N": "NO3",
        "Mo (µg/l)": "Mo",
        "Mo (ug/l)": "Mo",
        "PO4P (mg/l)": "PO4P",
    })
    observed["year"] = pd.to_numeric(observed["year"], errors="coerce").round(1)

    id_columns = [col for col in ["year", "Season", "Date Range", "n"] if col in observed.columns]
    value_columns = [spec.get("observed_column", parameter) for parameter, spec in PARAM_BLOCKS.items() if spec.get("observed_column", parameter) in observed.columns]
    observed_long = observed.melt(
        id_vars=id_columns,
        value_vars=value_columns,
        var_name="parameter",
        value_name="observed_conc",
    )
    observed_name_to_parameter = {spec.get("observed_column", parameter): parameter for parameter, spec in PARAM_BLOCKS.items()}
    observed_long["parameter"] = observed_long["parameter"].map(observed_name_to_parameter).fillna(observed_long["parameter"])
    observed_long["observed_conc"] = pd.to_numeric(observed_long["observed_conc"], errors="coerce")
    observed_long = observed_long[
        observed_long["year"].between(HINDCAST_START_YEAR, HINDCAST_END_YEAR, inclusive="both") &
        observed_long["observed_conc"].notna()
    ].copy()
    return observed_long.sort_values(["parameter", "year"]).reset_index(drop=True)


def load_lk_mix_schedule(path: Path, years) -> pd.DataFrame:
    """Load seasonal LK ratios from the ORE MIXES sheet.

    The sheet stores Kiruna in column E and Leveaniemi in column F for the seasonal
    rows. Missing later half-year rows are carried forward and flagged.
    """
    raw = pd.read_excel(path, sheet_name="ORE MIXES ", header=None, engine="openpyxl")
    seasonal = raw.iloc[:, [1, 4, 5]].copy()
    seasonal.columns = ["year", "lk_kiruna", "lk_leveaniemi"]
    for col in seasonal.columns:
        seasonal[col] = pd.to_numeric(seasonal[col], errors="coerce")
    seasonal = seasonal.dropna(subset=["year", "lk_kiruna", "lk_leveaniemi"])
    seasonal = seasonal[seasonal["year"].between(HINDCAST_START_YEAR, HINDCAST_END_YEAR, inclusive="both")]
    seasonal["year"] = seasonal["year"].round(1)
    seasonal["mix_from_file"] = True

    schedule = pd.DataFrame({"year": np.round(np.asarray(years, dtype=float), 1)})
    schedule = schedule.merge(seasonal, on="year", how="left")
    schedule["mix_from_file"] = schedule["mix_from_file"].fillna(False)
    schedule[["lk_kiruna", "lk_leveaniemi"]] = schedule[["lk_kiruna", "lk_leveaniemi"]].ffill().bfill()

    internal_total = schedule["lk_kiruna"] + schedule["lk_leveaniemi"]
    schedule["lk_kiruna"] = np.where(internal_total > 0, schedule["lk_kiruna"] / internal_total, LK_DEFAULT_KIRUNA_FRACTION)
    schedule["lk_leveaniemi"] = np.where(internal_total > 0, schedule["lk_leveaniemi"] / internal_total, LK_DEFAULT_LEVEANIEMI_FRACTION)
    schedule["gm"] = 0.0
    schedule["gk"] = 0.0
    schedule["gl"] = 0.0
    schedule["lk"] = 1.0
    return schedule[["year", "gm", "gk", "gl", "lk", "lk_leveaniemi", "lk_kiruna", "mix_from_file"]]


def load_production_reference(path: Path) -> pd.DataFrame:
    """Load production values for reference and reporting.

    The notebook displays these values but does not force them into the hindcast,
    because 2020 and 2021 are zero/missing in the provided sheet and the half-year
    split needs confirmation before they should replace the consultant template.
    """
    production = pd.read_excel(path, sheet_name="PRODUCTION", engine="openpyxl")
    production = production.rename(columns={production.columns[0]: "calendar_year"})
    production["calendar_year"] = pd.to_numeric(production["calendar_year"], errors="coerce")
    numeric_cols = [col for col in production.columns if col != "calendar_year"]
    for col in numeric_cols:
        production[col] = pd.to_numeric(production[col], errors="coerce")
    return production.dropna(subset=["calendar_year"]).reset_index(drop=True)


observed_long = load_observed_decimal_date(PARAMETERS_WORKBOOK_PATH)
HINDCAST_YEARS = np.sort(observed_long["year"].dropna().unique())
lk_hindcast_mix_schedule = load_lk_mix_schedule(PARAMETERS_WORKBOOK_PATH, HINDCAST_YEARS)
production_reference = load_production_reference(PARAMETERS_WORKBOOK_PATH)

print(f"Hindcast years found in observed data: {HINDCAST_YEARS}")
if not lk_hindcast_mix_schedule["mix_from_file"].all():
    carried = lk_hindcast_mix_schedule.loc[~lk_hindcast_mix_schedule["mix_from_file"], "year"].tolist()
    print("LK mix was missing for these years and was carried forward/backward:", carried)

display(lk_hindcast_mix_schedule)
display(observed_long.head(12))
display(production_reference)


## Method 1 Hindcast Validation and Export

This completes the deterministic consultant-formula approach. It predicts the observed 2020-2025 LK period, overlays the actual data as a separate line, calculates errors, and writes a Method 1 workbook. You can stop after this section and still have a complete non-ML result.


In [ ]:
def _safe_r2(y_true: pd.Series, y_pred: pd.Series) -> float:
    return _r2_score(y_true, y_pred)


def compare_hindcast_to_observations(summary: pd.DataFrame, observed: pd.DataFrame, method_name: str) -> pd.DataFrame:
    comparison = summary.merge(observed, on=["parameter", "year"], how="left")
    comparison["method"] = method_name
    comparison["error_p50_minus_observed"] = comparison["p50"] - comparison["observed_conc"]
    comparison["absolute_error"] = comparison["error_p50_minus_observed"].abs()
    comparison["percent_error"] = np.where(
        comparison["observed_conc"].abs() > 0,
        100 * comparison["error_p50_minus_observed"] / comparison["observed_conc"],
        np.nan,
    )
    comparison["observed_inside_p10_p90"] = comparison["observed_conc"].between(comparison["p10"], comparison["p90"])
    return comparison


def calculate_hindcast_metrics(comparison: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (method, parameter, unit), group in comparison.groupby(["method", "parameter", "unit"], dropna=False):
        valid = group.dropna(subset=["observed_conc", "p50"])
        rows.append({
            "method": method,
            "parameter": parameter,
            "unit": unit,
            "n_observed": int(valid["observed_conc"].notna().sum()),
            "mae": _mean_absolute_error(valid["observed_conc"], valid["p50"]) if len(valid) else np.nan,
            "mean_abs_percent_error": valid["percent_error"].abs().mean() if len(valid) else np.nan,
            "r2_observed_vs_p50": _safe_r2(valid["observed_conc"], valid["p50"]),
            "p10_p90_coverage": valid["observed_inside_p10_p90"].mean() if len(valid) else np.nan,
        })
    return pd.DataFrame(rows)


# Method 1: deterministic consultant formula hindcast.
consultant_formula_hindcast_simulations = pd.concat(
    [
        consultant_formula_monte_carlo(
            parameter,
            model_data,
            scenario=lk_hindcast_mix_schedule.drop(columns=["mix_from_file"]),
            ore_inputs=None,
            n_runs=HINDCAST_MONTE_CARLO_RUNS,
            random_state=RANDOM_STATE + 8000,
            prediction_years=HINDCAST_YEARS,
        )
        for parameter in PARAM_BLOCKS
    ],
    ignore_index=True,
)
consultant_formula_hindcast_summary = summarize_consultant_formula_simulations(consultant_formula_hindcast_simulations)
consultant_formula_hindcast_comparison = compare_hindcast_to_observations(
    consultant_formula_hindcast_summary,
    observed_long,
    CONSULTANT_FORMULA_METHOD,
)
consultant_formula_hindcast_metrics = calculate_hindcast_metrics(consultant_formula_hindcast_comparison)

display(consultant_formula_hindcast_metrics)
display(consultant_formula_hindcast_comparison.sort_values(["parameter", "year"]).head(32))


In [ ]:
def plot_hindcast_validation(
    historical_data: pd.DataFrame,
    hindcast: pd.DataFrame,
    observed: pd.DataFrame,
    method_name: str,
    out_filename: str,
    diagnostics: pd.DataFrame | None = None,
    band_color: str = "#93c5fd",
    line_color: str = "#2563eb",
) -> Path:
    fig, axes = make_parameter_axes(len(PARAM_BLOCKS), panel_height=3.6)

    for ax, parameter in zip(axes, PARAM_BLOCKS):
        pred = hindcast[hindcast["parameter"] == parameter].sort_values("year")
        obs = observed[observed["parameter"] == parameter].sort_values("year")
        unit = PARAM_BLOCKS[parameter]["unit"]

        title_suffix = ""
        if diagnostics is not None and not diagnostics[diagnostics["parameter"] == parameter].empty:
            diag = diagnostics[diagnostics["parameter"] == parameter].iloc[0]
            r2 = diag["half_year_cv_r2"] if diag["forecast_model"] == "separate half-year RF" else diag["full_cv_r2"]
            title_suffix = f" | surrogate CV R²={r2:.2f}"

        ax.fill_between(pred["year"].to_numpy(), pred["p10"].to_numpy(), pred["p90"].to_numpy(), color=band_color, alpha=0.35, label="Hindcast P10-P90")
        ax.plot(pred["year"], pred["p50"], color=line_color, marker="o", linewidth=2.0, markersize=4, label="Hindcast P50")
        ax.plot(obs["year"], obs["observed_conc"], color="#b91c1c", marker="s", linewidth=2.0, markersize=4, zorder=4, label="Observed actual")
        ax.set_title(f"{parameter} ({unit}){title_suffix}")
        ax.set_ylabel(f"Concentration ({unit})")
        ax.grid(True, alpha=0.25)

    for ax in axes:
        ax.set_xlabel("Year")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
    fig.suptitle(f"2020-2025 LK Hindcast Validation: {method_name}", y=0.98, fontsize=15)
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))

    out_path = OUTPUT_DIR / out_filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    return out_path


consultant_formula_hindcast_figure_path = plot_hindcast_validation(
    model_data,
    consultant_formula_hindcast_summary,
    observed_long,
    method_name=CONSULTANT_FORMULA_METHOD,
    out_filename="leveaniemi_hindcast_validation_consultant_formula_2020_2025.png",
    diagnostics=None,
    band_color="#f9a8d4",
    line_color="#be185d",
)

method1_path = OUTPUT_DIR / "leveaniemi_method1_consultant_formula_outputs.xlsx"
element_limits = pd.DataFrame([
    {"parameter": parameter, "unit": PARAM_BLOCKS[parameter]["unit"], "limit": limit}
    for parameter, limit in ELEMENT_LIMITS.items()
])
forecast_input_note = pd.DataFrame([
    {
        "input_mode": FORECAST_INPUT_MODE,
        "selected_proxy_scenario": SELECTED_ORE_MIX_LABEL if NEW_ORE_INPUTS is None else None,
        "note": "Workbook GK/GL values are consultant-derived proxy/sensitivity inputs, and LK is inferred from Leveaniemi and Kiruna proxy rates because the consultant did not include LK directly. LK ratios can vary by year/season through TIME_VARYING_ORE_MIX. These are not measured current/future ore-plan truth unless confirmed by the mine. Provide NEW_ORE_INPUTS for final new-ore forecasts.",
    }
])

with pd.ExcelWriter(method1_path) as writer:
    consultant_formula_reproduction.to_excel(writer, sheet_name="Formula_reproduction", index=False)
    consultant_formula_reproduction_metrics.to_excel(writer, sheet_name="Formula_repro_metrics", index=False)
    consultant_formula_forecast_summary.to_excel(writer, sheet_name="Formula_forecast", index=False)
    consultant_formula_annual_forecast_summary.to_excel(writer, sheet_name="Formula_forecast_annual", index=False)
    selected_mix_table.to_excel(writer, sheet_name="Selected_proxy_mix", index=False)
    lk_hindcast_mix_schedule.to_excel(writer, sheet_name="LK_mix_schedule", index=False)
    production_reference.to_excel(writer, sheet_name="Production_reference", index=False)
    observed_long.to_excel(writer, sheet_name="Observed_decimal_date", index=False)
    consultant_formula_hindcast_summary.to_excel(writer, sheet_name="Formula_hindcast", index=False)
    consultant_formula_hindcast_comparison.to_excel(writer, sheet_name="Formula_comparison", index=False)
    consultant_formula_hindcast_metrics.to_excel(writer, sheet_name="Formula_metrics", index=False)
    element_limits.to_excel(writer, sheet_name="Element_limits", index=False)
    forecast_input_note.to_excel(writer, sheet_name="Forecast_input_note", index=False)

print(f"Saved consultant-formula forecast figure: {consultant_formula_figure_path}")
print(f"Saved consultant-formula hindcast figure: {consultant_formula_hindcast_figure_path}")
print(f"Saved Method 1 workbook: {method1_path}")


## Optional Method 2: Machine-Learning Surrogate

Run the remaining cells only if you want the Random Forest/data-science approach as a second method. Method 2 learns the consultant workbook recurrence and then repeats the forecast and 2020-2025 validation using the ML surrogate.


### Method 2A: Model Training and Diagnostics

The Random Forest is trained separately for each contaminant. Features are imputed and standardized within each parameter block so chloride's mg/kg-scale leaching values do not dominate the other blocks. If a full model has weak cross-validated R², the notebook also evaluates separate winter/summer models and uses them if they improve the diagnostic score.


In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import mean_absolute_error, r2_score
    from sklearn.model_selection import KFold, cross_val_predict
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Optional Method 2 requires scikit-learn. Install it with `%pip install scikit-learn`, "
        "or stop after Method 1 if you only need the consultant-formula approach."
    ) from exc

FEATURE_COLUMNS = [
    "year_index",
    "calendar_year",
    "half_year",
    "tailings_load",
    "ore_frac_gm",
    "ore_frac_gk",
    "ore_frac_gl",
    "ore_frac_lk",
    "production_mton",
    "process_leach",
    "leach_per_mton",
    "leach_per_storage_volume",
    "pit_pump_volume",
    "pit_pump_conc",
    "pump_conc_x_volume",
    "gross_gain_flow",
    "gross_loss_flow",
    "net_flow",
    "known_water_load_proxy",
    "storage_volume",
    STATE_COLUMN,
    "storage_mass_proxy",
    "losses",
    "gains",
]


def make_model(seed: int = RANDOM_STATE) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("rf", RandomForestRegressor(
                n_estimators=RF_N_ESTIMATORS,
                min_samples_leaf=1,
                max_features="sqrt",
                bootstrap=True,
                random_state=seed,
                n_jobs=-1,
            )),
        ]
    )


def cv_predict(model: Pipeline, X: pd.DataFrame, y: pd.Series, max_splits: int = 5) -> np.ndarray:
    n = len(y)
    if n < 4:
        return np.full(n, np.nan)
    splits = min(max_splits, n)
    cv = KFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_predict(model, X, y, cv=cv, n_jobs=None)


def evaluate_half_year_models(train: pd.DataFrame) -> tuple[float, dict[int, Pipeline], pd.Series]:
    preds = pd.Series(index=train.index, dtype=float)
    models: dict[int, Pipeline] = {}
    for half_value in [0, 1]:
        sub = train[train["half_year"] == half_value]
        if len(sub) < 4:
            continue
        X_sub = sub[FEATURE_COLUMNS]
        y_sub = sub[TARGET_COLUMN]
        preds.loc[sub.index] = cv_predict(make_model(RANDOM_STATE + half_value + 10), X_sub, y_sub, max_splits=3)
        model = make_model(RANDOM_STATE + half_value + 100)
        model.fit(X_sub, y_sub)
        models[half_value] = model
    if preds.notna().all():
        return r2_score(train[TARGET_COLUMN], preds.loc[train.index]), models, preds
    return np.nan, models, preds


def train_parameter_models(data: pd.DataFrame, train_end_year: float = TRAIN_END_YEAR):
    model_packs = {}
    diagnostic_rows = []
    cv_rows = []

    for parameter, param_df in data.groupby("parameter", sort=False):
        train = param_df[
            (~param_df["is_initial_row"]) &
            (param_df["year"] <= train_end_year) &
            param_df[TARGET_COLUMN].notna()
        ].copy()
        train = train.dropna(subset=[TARGET_COLUMN])
        X = train[FEATURE_COLUMNS]
        y = train[TARGET_COLUMN]

        full_model_for_cv = make_model(RANDOM_STATE)
        full_cv_pred = cv_predict(full_model_for_cv, X, y, max_splits=5)
        full_r2 = r2_score(y, full_cv_pred) if np.isfinite(full_cv_pred).all() else np.nan
        full_mae = mean_absolute_error(y, full_cv_pred) if np.isfinite(full_cv_pred).all() else np.nan

        half_r2, half_models, half_preds = evaluate_half_year_models(train)
        use_half_models = bool(np.isfinite(half_r2) and (not np.isfinite(full_r2) or half_r2 > full_r2 + 0.03) and full_r2 < 0.70)

        final_model = make_model(RANDOM_STATE)
        final_model.fit(X, y)

        model_packs[parameter] = {
            "parameter": parameter,
            "unit": train["unit"].iloc[0],
            "full_model": final_model,
            "half_models": half_models,
            "use_half_models": use_half_models,
            "training_rows": len(train),
            "full_cv_r2": full_r2,
            "full_cv_mae": full_mae,
            "half_cv_r2": half_r2,
        }

        diagnostic_rows.append({
            "parameter": parameter,
            "unit": train["unit"].iloc[0],
            "training_rows": len(train),
            "full_cv_r2": full_r2,
            "full_cv_mae": full_mae,
            "half_year_cv_r2": half_r2,
            "forecast_model": "separate half-year RF" if use_half_models else "single RF with half_year feature",
        })

        for idx, row in train.iterrows():
            cv_rows.append({
                "parameter": parameter,
                "year": row["year"],
                "observed": row[TARGET_COLUMN],
                "cv_pred_single_rf": full_cv_pred[list(train.index).index(idx)] if np.isfinite(full_cv_pred).all() else np.nan,
                "cv_pred_half_year_rf": half_preds.loc[idx] if idx in half_preds.index else np.nan,
            })

    diagnostics = pd.DataFrame(diagnostic_rows).sort_values("parameter")
    cv_predictions = pd.DataFrame(cv_rows).sort_values(["parameter", "year"])
    return model_packs, diagnostics, cv_predictions


model_packs, diagnostics, cv_predictions = train_parameter_models(model_data)
display(diagnostics)

### Method 2B: Sequential ML Forecast and Monte Carlo Uncertainty

This is the ML equivalent of the consultant-formula forecast. The predicted concentration is fed forward as the next storage state, and leaching is perturbed by ±15% for uncertainty bands.


In [ ]:
def predict_array_from_pack(model_pack: dict, feature_rows: pd.DataFrame) -> np.ndarray:
    """Predict many rows at once while respecting optional half-year submodels."""
    if model_pack["use_half_models"]:
        predictions = pd.Series(index=feature_rows.index, dtype=float)
        for half_value, sub_index in feature_rows.groupby("half_year").groups.items():
            model = model_pack["half_models"].get(int(half_value), model_pack["full_model"])
            predictions.loc[sub_index] = model.predict(feature_rows.loc[sub_index, FEATURE_COLUMNS])
        values = predictions.to_numpy(dtype=float)
    else:
        values = model_pack["full_model"].predict(feature_rows[FEATURE_COLUMNS]).astype(float)
    return np.maximum(values, 0.0)


def predict_from_pack(model_pack: dict, feature_row: pd.DataFrame) -> float:
    return float(predict_array_from_pack(model_pack, feature_row)[0])


def sequential_forecast(
    parameter: str,
    data: pd.DataFrame,
    model_pack: dict,
    scenario: str | dict | pd.DataFrame = SELECTED_ORE_MIX_SCENARIO,
    ore_inputs=NEW_ORE_INPUTS,
    n_runs: int = N_MONTE_CARLO,
    leach_perturbation: float = LEACH_PERTURBATION,
    random_state: int = RANDOM_STATE,
    prediction_years=None,
) -> pd.DataFrame:
    param_df = data[data["parameter"] == parameter].sort_values("year").copy()
    prediction_years = FORECAST_YEARS if prediction_years is None else np.round(np.asarray(prediction_years, dtype=float), 1)
    forecast_template = param_df[param_df["year"].isin(prediction_years)].copy()
    if forecast_template.empty:
        raise ValueError(f"No template rows found for {parameter} and years {prediction_years}.")
    forecast_template = apply_ore_mix(forecast_template, scenario, ore_inputs=ore_inputs).sort_values("year")
    input_mode = (
        "user_supplied_new_ore_inputs" if ore_inputs is not None
        else "time_varying_proxy_ore_mix" if isinstance(scenario, pd.DataFrame)
        else "consultant_proxy_sensitivity"
    )

    start_year = float(np.min(prediction_years))
    historical_state = param_df[(param_df["year"] < start_year) & param_df[TARGET_COLUMN].notna()].sort_values("year")
    if historical_state.empty:
        raise ValueError(f"Cannot start recurrence for {parameter}; no previous state exists before {start_year}.")
    start_state = float(historical_state[TARGET_COLUMN].dropna().iloc[-1])

    rng = np.random.default_rng(random_state + sum(ord(ch) for ch in parameter))
    run_ids = np.arange(n_runs)
    states = np.full(n_runs, start_state, dtype=float)
    record_frames = []

    for _, template_row in forecast_template.iterrows():
        batch = pd.DataFrame([template_row.to_dict()] * n_runs)
        batch[STATE_COLUMN] = states
        perturbations = rng.uniform(1.0 - leach_perturbation, 1.0 + leach_perturbation, size=n_runs)
        batch["process_leach"] = pd.to_numeric(batch["process_leach"], errors="coerce").to_numpy(dtype=float) * perturbations
        feature_rows = add_derived_features(batch)
        predictions = predict_array_from_pack(model_pack, feature_rows)
        states = predictions

        record_frames.append(pd.DataFrame({
            "run": run_ids,
            "input_mode": input_mode,
            "parameter": parameter,
            "unit": model_pack["unit"],
            "year": float(template_row["year"]),
            "calendar_year": int(template_row["calendar_year"]),
            "half_year": int(template_row["half_year"]),
            "ore_frac_gm": float(template_row["ore_frac_gm"]),
            "ore_frac_gk": float(template_row["ore_frac_gk"]),
            "ore_frac_gl": float(template_row["ore_frac_gl"]),
            "ore_frac_lk": float(template_row["ore_frac_lk"]),
            "lk_leveaniemi_frac": float(template_row.get("lk_leveaniemi_frac", LK_DEFAULT_LEVEANIEMI_FRACTION)),
            "lk_kiruna_frac": float(template_row.get("lk_kiruna_frac", LK_DEFAULT_KIRUNA_FRACTION)),
            "prediction": predictions,
        }))

    return pd.concat(record_frames, ignore_index=True)


def summarize_forecast(simulations: pd.DataFrame) -> pd.DataFrame:
    group_columns = ["parameter", "unit", "year", "calendar_year", "half_year", "ore_frac_gm", "ore_frac_gk", "ore_frac_gl", "ore_frac_lk", "lk_leveaniemi_frac", "lk_kiruna_frac"]
    if "input_mode" in simulations.columns:
        group_columns.insert(2, "input_mode")
    summary = (
        simulations
        .groupby(group_columns, dropna=False)["prediction"]
        .quantile([0.10, 0.50, 0.90])
        .unstack()
        .rename(columns={0.10: "p10", 0.50: "p50", 0.90: "p90"})
        .reset_index()
        .sort_values(["parameter", "year"])
    )
    return summary


forecast_simulations = pd.concat(
    [sequential_forecast(parameter, model_data, model_packs[parameter]) for parameter in PARAM_BLOCKS],
    ignore_index=True,
)
forecast_summary = summarize_forecast(forecast_simulations)
annual_forecast_summary = forecast_summary[np.isclose(forecast_summary["year"] % 1, 0.0)].copy()

display(forecast_summary.head(12))
display(annual_forecast_summary)


### Method 2C: Proxy Ore-Combination Sensitivity

This optional sensitivity table compares the forecast response under several plausible GK/GL/LK ore-mix assumptions when confirmed future ore inputs are not available.


In [ ]:
def run_proxy_sensitivity(scenarios: dict | None = None, runs_per_mix: int = 200) -> pd.DataFrame:
    frames = []
    scenarios = scenarios or ORE_MIX_SCENARIOS
    for scenario_index, (scenario_name, scenario) in enumerate(scenarios.items()):
        sims = pd.concat(
            [
                sequential_forecast(
                    parameter,
                    model_data,
                    model_packs[parameter],
                    scenario=scenario,
                    ore_inputs=None,
                    n_runs=runs_per_mix,
                    random_state=RANDOM_STATE + scenario_index * 1000,
                )
                for parameter in PARAM_BLOCKS
            ],
            ignore_index=True,
        )
        summary = summarize_forecast(sims)
        normalized = normalize_mix(scenario)
        summary["sensitivity_scenario"] = scenario_name
        for component in ORE_COMPONENTS:
            summary[f"sensitivity_{component}_fraction"] = normalized[component]
        lk_normalized = normalize_lk_internal_mix(scenario)
        summary["sensitivity_lk_leveaniemi_fraction"] = lk_normalized["lk_leveaniemi"]
        summary["sensitivity_lk_kiruna_fraction"] = lk_normalized["lk_kiruna"]
        frames.append(summary)
    return pd.concat(frames, ignore_index=True)


sensitivity_summary = run_proxy_sensitivity()
sensitivity_annual = sensitivity_summary[np.isclose(sensitivity_summary["year"] % 1, 0.0)].copy()

display(sensitivity_annual.head(20))

### Method 2D: ML Forecast Figure


In [ ]:
def plot_forecast_panels(data: pd.DataFrame, forecast: pd.DataFrame, diagnostics: pd.DataFrame) -> Path:
    fig, axes = make_parameter_axes(len(PARAM_BLOCKS), panel_height=3.6)

    for ax, parameter in zip(axes, PARAM_BLOCKS):
        hist = data[(data["parameter"] == parameter) & data["year"].between(ACTUAL_DATA_START_YEAR, TRAIN_END_YEAR, inclusive="both")].sort_values("year")
        fc = forecast[forecast["parameter"] == parameter].sort_values("year")
        unit = PARAM_BLOCKS[parameter]["unit"]
        diag = diagnostics[diagnostics["parameter"] == parameter].iloc[0]
        r2 = diag["half_year_cv_r2"] if diag["forecast_model"] == "separate half-year RF" else diag["full_cv_r2"]

        ax.plot(
            hist["year"], hist[TARGET_COLUMN],
            color="#1f2937", marker="o", linewidth=1.8, markersize=4,
            label="Consultant workbook model, not observed"
        )
        ax.fill_between(
            fc["year"].to_numpy(), fc["p10"].to_numpy(), fc["p90"].to_numpy(),
            color="#6aaed6", alpha=0.28, label="P10-P90"
        )
        ax.plot(fc["year"], fc["p50"], color="#0b6e99", marker="o", linewidth=2.0, markersize=4, label="Forecast P50")
        limit = ELEMENT_LIMITS.get(parameter)
        if limit is not None and np.isfinite(float(limit)):
            ax.axhline(float(limit), color="#b91c1c", linestyle=":", linewidth=1.6, label="Limit")
        ax.axvline(TRAIN_END_YEAR, color="#9ca3af", linestyle="--", linewidth=1.2)
        ax.set_title(f"{parameter} ({unit}) | CV R²={r2:.2f}")
        ax.set_ylabel(f"Concentration ({unit})")
        ax.grid(True, alpha=0.25)

    for ax in axes:
        ax.set_xlabel("Year")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=False)
    input_label = "user-supplied new ore inputs" if NEW_ORE_INPUTS is not None else f"proxy ore mix: {SELECTED_ORE_MIX_LABEL}"
    fig.suptitle(f"Leveäniemi Process-Water Forecast ({input_label}) with ±15% Leaching Uncertainty", y=0.98, fontsize=15)
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))

    out_path = OUTPUT_DIR / "leveaniemi_forecast_bands.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    return out_path


figure_path = plot_forecast_panels(model_data, forecast_summary, diagnostics)
figure_path

### Method 2E: ML Forecast Export


In [ ]:
ml_forecast_path = OUTPUT_DIR / "leveaniemi_method2_ml_outputs.xlsx"

with pd.ExcelWriter(ml_forecast_path) as writer:
    diagnostics.to_excel(writer, sheet_name="ML_diagnostics", index=False)
    cv_predictions.to_excel(writer, sheet_name="ML_CV_predictions", index=False)
    forecast_summary.to_excel(writer, sheet_name="ML_forecast_half_year", index=False)
    annual_forecast_summary.to_excel(writer, sheet_name="ML_forecast_annual", index=False)
    sensitivity_annual.to_excel(writer, sheet_name="ML_sensitivity_annual", index=False)
    consultant_formula_forecast_summary.to_excel(writer, sheet_name="Formula_forecast_ref", index=False)
    consultant_formula_annual_forecast_summary.to_excel(writer, sheet_name="Formula_forecast_annual_ref", index=False)
    selected_mix_table.to_excel(writer, sheet_name="Selected_proxy_mix", index=False)
    element_limits.to_excel(writer, sheet_name="Element_limits", index=False)
    forecast_input_note.to_excel(writer, sheet_name="Forecast_input_note", index=False)

print(f"Saved ML forecast figure: {figure_path}")
print(f"Saved Method 2 ML workbook: {ml_forecast_path}")


## Optional: Join Daily Monitoring Data

If daily monitoring data is available later, load it here, aggregate it by year or season, and plot it against `forecast_summary`. Keep it separate from training unless the thesis question changes from reproducing the consultant's recurrence to calibrating against observations.

In [ ]:
# Example scaffold for later monitoring-data comparison:
# monitoring_path = Path("daily_monitoring.xlsx")
# monitoring = pd.read_excel(monitoring_path)
# monitoring["date"] = pd.to_datetime(monitoring["date"])
# monitoring["year"] = monitoring["date"].dt.year + np.where(monitoring["date"].dt.month >= 6, 0.5, 0.0)
# seasonal_monitoring = monitoring.groupby(["parameter", "year"], as_index=False)["concentration"].median()
# display(seasonal_monitoring.head())

### Method 2F: ML Hindcast Validation

This repeats the 2020-2025 validation using the Random Forest surrogate. It is intentionally placed after the ML forecast section so the non-ML consultant method remains complete without it.


In [ ]:
HINDCAST_TRAIN_END_YEAR = 2019.5

# Method 2: Random Forest surrogate hindcast.
hindcast_model_packs, hindcast_diagnostics, hindcast_cv_predictions = train_parameter_models(
    model_data,
    train_end_year=HINDCAST_TRAIN_END_YEAR,
)

hindcast_simulations = pd.concat(
    [
        sequential_forecast(
            parameter,
            model_data,
            hindcast_model_packs[parameter],
            scenario=lk_hindcast_mix_schedule.drop(columns=["mix_from_file"]),
            ore_inputs=None,
            n_runs=HINDCAST_MONTE_CARLO_RUNS,
            random_state=RANDOM_STATE + 5000,
            prediction_years=HINDCAST_YEARS,
        )
        for parameter in PARAM_BLOCKS
    ],
    ignore_index=True,
)

hindcast_summary = summarize_forecast(hindcast_simulations)
hindcast_summary["method"] = "ML Random Forest surrogate"
hindcast_comparison = compare_hindcast_to_observations(
    hindcast_summary,
    observed_long,
    "ML Random Forest surrogate",
)
hindcast_metrics = calculate_hindcast_metrics(hindcast_comparison)
combined_hindcast_metrics = pd.concat([consultant_formula_hindcast_metrics, hindcast_metrics], ignore_index=True)
combined_hindcast_comparison = pd.concat([consultant_formula_hindcast_comparison, hindcast_comparison], ignore_index=True)

display(hindcast_diagnostics)
display(hindcast_metrics)
display(combined_hindcast_comparison.sort_values(["method", "parameter", "year"]).head(32))


### Method 2G: ML Hindcast Figure and Combined Export

The ML chart uses the same observed 2020-2025 line as Method 1. The combined workbook is useful when you want to compare the deterministic consultant formula and ML surrogate side by side.


In [ ]:
hindcast_figure_path = plot_hindcast_validation(
    model_data,
    hindcast_summary,
    observed_long,
    method_name="ML Random Forest surrogate",
    out_filename="leveaniemi_hindcast_validation_ml_2020_2025.png",
    diagnostics=hindcast_diagnostics,
    band_color="#93c5fd",
    line_color="#2563eb",
)
hindcast_path = OUTPUT_DIR / "leveaniemi_hindcast_validation_2020_2025.xlsx"

with pd.ExcelWriter(hindcast_path) as writer:
    lk_hindcast_mix_schedule.to_excel(writer, sheet_name="LK_mix_schedule", index=False)
    production_reference.to_excel(writer, sheet_name="Production_reference", index=False)
    observed_long.to_excel(writer, sheet_name="Observed_decimal_date", index=False)
    consultant_formula_hindcast_summary.to_excel(writer, sheet_name="Formula_hindcast", index=False)
    consultant_formula_hindcast_comparison.to_excel(writer, sheet_name="Formula_comparison", index=False)
    consultant_formula_hindcast_metrics.to_excel(writer, sheet_name="Formula_metrics", index=False)
    hindcast_diagnostics.to_excel(writer, sheet_name="ML_diagnostics_pre2020", index=False)
    hindcast_cv_predictions.to_excel(writer, sheet_name="ML_CV_predictions", index=False)
    hindcast_summary.to_excel(writer, sheet_name="ML_hindcast", index=False)
    hindcast_comparison.to_excel(writer, sheet_name="ML_comparison", index=False)
    hindcast_metrics.to_excel(writer, sheet_name="ML_metrics", index=False)
    combined_hindcast_metrics.to_excel(writer, sheet_name="Combined_metrics", index=False)
    combined_hindcast_comparison.to_excel(writer, sheet_name="Combined_comparison", index=False)

print(f"Saved ML hindcast figure: {hindcast_figure_path}")
print(f"Saved combined Method 1 + Method 2 hindcast workbook: {hindcast_path}")
